# Assemble a simulated trader from tools and resources: practice

Configure researcher/trader servers → turn researcher into a tool → read account and strategy resources → run trader → migrate orchestration into modules.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [6_mcp/4_lab4.ipynb](../../../../../6_mcp/4_lab4.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. MCP tools, resources, and server lifetime

A tool performs an operation; a resource supplies readable context. A client discovers tool descriptions and calls them while the server is connected. This fixture is not a protocol transport.

**Predict:** Is listing tool names the same as executing them?

In [ ]:
from contextlib import contextmanager
calls = []
@contextmanager
def practice_server():
    tools = {"add": lambda a, b: a + b}
    try:
        calls.append("connected")
        yield tools
    finally:
        calls.append("closed")
with practice_server() as tools:
    print("Discovered:", list(tools))
    result = tools["add"](12, 30)
    print("Called add:", result)
assert result == 42 and calls == ["connected", "closed"]

**Check:** No. Discovery returns descriptions. A real stdio subprocess also has a working directory, environment, and protocol stream.

**Change one thing:** Add a read-only dictionary resource and a tool returning a fixed date. Explain how discovery differs from invocation.

## 2. Plan → search → synthesize

The research workflow separates a search plan, collected findings, and a report. Use fixed fictional snippets first so you can inspect how evidence reaches the writer. No current facts are being researched here.

**Predict:** Will the report receive the original query as well as the findings?

In [ ]:
from dataclasses import dataclass
@dataclass
class SearchItem:
    reason: str
    query: str
query = "How should a beginner practice tool calls?"
plan = [SearchItem("Understand arguments", "tool argument schema"),
        SearchItem("Understand execution", "tool dispatcher")]
fixture = {"tool argument schema": "A schema describes arguments.",
           "tool dispatcher": "Application code executes an allowed function."}
findings = [fixture[item.query] for item in plan]
report = {"query": query, "summary": " ".join(findings), "follow_up_questions": ["How do we test invalid arguments?"]}
print(report)
assert len(findings) == 2 and "Application" in report["summary"]

**Check:** Yes. Plans choose questions, search collects material, and synthesis uses it. Delivery is a separate side effect.

**Change one thing:** Add a third search plan item and a matching fixture. Explain how the original query keeps synthesis focused.

## 3. Agent as a tool vs. handoff

A tool returns its result to the calling agent. A handoff transfers control to the receiving agent. This trace uses ordinary Python to make that distinction visible.

**Predict:** After the tool finishes, who produces the next step?

In [ ]:
def specialist_tool(topic):
    return f"Specialist result for {topic}"
tool_trace = ["manager starts", specialist_tool("Python"), "manager resumes"]
handoff_trace = ["manager starts", "manager hands off", "specialist controls remaining reply"]
print("Tool:", tool_trace)
print("Handoff:", handoff_trace)
assert tool_trace[-1] == "manager resumes"

**Check:** The manager resumes after a tool. The specialist takes control after a handoff.

**Change one thing:** Sketch which pattern fits a writer consulting a researcher, and which fits routing a customer to support.

## 4. Simulated accounts change state

Buying shares updates cash, holdings, and a transaction record. Practice against a dictionary so you do not import account modules that create or alter databases.

**Predict:** How much cash remains after buying three fictional shares at 100 each?

In [ ]:
account = {"cash": 1000.0, "holdings": {}, "transactions": []}
def buy(symbol, quantity, price):
    cost = quantity * price
    if quantity <= 0 or price <= 0 or cost > account["cash"]:
        raise ValueError("Invalid or unaffordable order")
    account["cash"] -= cost
    account["holdings"][symbol] = account["holdings"].get(symbol, 0) + quantity
    account["transactions"].append({"symbol": symbol, "quantity": quantity, "price": price})
buy("DEMO", 3, 100)
print(account)
assert account["cash"] == 700 and account["holdings"]["DEMO"] == 3

**Check:** 700. These are fictional values and this code places no real order.

**Change one thing:** Try an unaffordable order and confirm the balance is unchanged. Explain why reset is a data-changing action.

## 5. Cache and simulation are not live prices

The trading exercises use market data and cached values. A fixed fictional quote lets you observe caching. A fallback simulation must stay identifiable as simulated data.

**Predict:** How many provider-fixture calls occur for the same cached symbol?

In [ ]:
from functools import lru_cache
calls = []
@lru_cache(maxsize=8)
def practice_quote(symbol):
    calls.append(symbol)
    return {"symbol": symbol, "price": 100.0, "source": "fictional fixture"}
first = practice_quote("DEMO")
second = practice_quote("DEMO")
print(first, "Fixture calls:", len(calls))
assert len(calls) == 1 and second["source"] == "fictional fixture"

**Check:** One. Cached or random fallback prices are not verified current market values.

**Change one thing:** Call a second symbol. Explain how you would record a quote timestamp and expire old cache entries.

## 6. Await vs. concurrent work

Calling an async function creates a coroutine. Await executes/waits for it. gather schedules independent coroutines and returns results in input order. Notebook cells support top-level await.

**Predict:** Will results follow completion order or input order?

In [ ]:
import asyncio
async def work(label, delay):
    await asyncio.sleep(delay)  # Yield control while waiting.
    return label
results = await asyncio.gather(work("A", 0.002), work("B", 0))
print(results)
assert results == ["A", "B"]

**Check:** Input order. In a standalone script, wrap work in async def main and call asyncio.run(main()); do not do that inside a running notebook loop.

**Change one thing:** Run the two functions with separate awaits. Explain why a dependent stage cannot start until it has its input.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.